# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [6]:
import json
import sys
import numpy as np
import pandas as pd

import catboost as cb

from sklearn.model_selection import KFold

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")

y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)

features = tr.columns.drop("daily_orders").to_list()
cat_features = tr.select_dtypes(include=['object', 'category']).columns.tolist()

X = tr[features]
X_test = te[features]

splits = list(KFold(4, shuffle=True, random_state=0).split(tr))

oof = np.zeros((len(tr), 3))
test_preds = np.zeros((len(te), 3))

cb_params = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 0,
    'random_seed': 42
}


for i, q in enumerate(QUANTILES):
    print(f"Обучение моделей для квантиля {q}...")

    fold_test_preds = np.zeros(len(te))

    for train_idx, val_idx in splits:
        X_train, y_train = X.iloc[train_idx], y[train_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]

        train_pool = cb.Pool(X_train, y_train, cat_features=cat_features)
        val_pool = cb.Pool(X_val, y_val, cat_features=cat_features)

        model = cb.CatBoostRegressor(loss_function=f'Quantile:alpha={q}', **cb_params)

        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)

        oof[val_idx, i] = model.predict(val_pool)

        fold_test_preds += model.predict(cb.Pool(X_test, cat_features=cat_features)) / len(splits)

    test_preds[:, i] = fold_test_preds


oof = np.maximum(oof, 0)
oof[:, 1] = np.maximum(oof[:, 1], oof[:, 0])
oof[:, 2] = np.maximum(oof[:, 2], oof[:, 1])

test_preds = np.maximum(test_preds, 0)
test_preds[:, 1] = np.maximum(test_preds[:, 1], test_preds[:, 0])
test_preds[:, 2] = np.maximum(test_preds[:, 2], test_preds[:, 1])

oof = np.minimum(oof, 5000)
test_preds = np.minimum(test_preds, 5000)

folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]

sub = pd.DataFrame({
    "id": te["id"],
    "predicted_daily_orders_p10": np.round(test_preds[:, 0], 1),
    "predicted_daily_orders_p50": np.round(test_preds[:, 1], 1),
    "predicted_daily_orders_p90": np.round(test_preds[:, 2], 1)
})

assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("catboost_baseline", [f / 100 for f in folds])
print(json.dumps(cv))


Обучение моделей для квантиля 0.1...
Обучение моделей для квантиля 0.5...
Обучение моделей для квантиля 0.9...
{"name": "catboost_baseline", "metric": "quantile_quality_x100", "folds": [75.4, 76.84, 76.15, 77.0], "cv": 76.35, "std": 0.63}
